In [1]:
import os
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

DATA_PATH  = "processed/floodsense_onset.csv"
MODELS_DIR = "final_models"
os.makedirs(MODELS_DIR, exist_ok=True)

N_TRIALS = 40
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["district", "date"]).reset_index(drop=True)

exclude_cols = ["date", "district", "flood_event", "flood_onset", "flood_state",
                "event_num", "event_id"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

groups = df["district"]
N = len(df)

print(f"Loaded {df.shape[0]} rows, using {len(feature_cols)} features")
df.head()

Loaded 1365 rows, using 53 features


,date,evaporation,precipitation,pressure,soil_moisture,temperature,wind_speed,humidity,month,is_monsoon,...,water_area_km2_lag3_zscore,water_area_km2_lag5_zscore,water_area_km2_lag7_zscore,water_area_km2_lag14_zscore,days_since_heavy_rain,days_since_dry_day,days_in_monsoon,precip_x_soil_moisture,precip_7day_x_monsoon,pressure_low_x_humidity_high
0,2023-01-01,-0.000230,0.000135,100639.5491,0.087776,16.712126,0.625041,50.787158,1,0,...,NaN,NaN,NaN,NaN,NaN,0.0,0,0.000012,0.0,0.0
1,2023-01-01,-0.000230,NaN,100639.5491,0.087776,16.712126,0.625041,50.787158,1,0,...,NaN,NaN,NaN,NaN,NaN,0.0,0,NaN,0.0,0.0
2,2023-01-02,-0.000215,0.000000,100800.0680,0.087691,16.070381,2.556115,54.975463,1,0,...,NaN,NaN,NaN,NaN,NaN,0.0,0,0.000000,0.0,0.0
3,2023-01-03,-0.000217,0.000000,101029.1779,0.087694,15.405261,2.904837,49.248183,1,0,...,-1.187057,NaN,NaN,NaN,NaN,0.0,0,0.000000,0.0,0.0
4,2023-01-04,-0.000214,0.000000,101026.6490,0.087631,14.635752,2.287333,51.012060,1,0,...,-0.449849,NaN,NaN,NaN,NaN,0.0,0,0.000000,0.0,0.0


In [3]:
def make_objective(target, training_filter=None):
    def _objective(trial):
        params = {
            "objective":        "binary",
            "metric":           "auc",
            "learning_rate":    0.05,
            "num_leaves":       trial.suggest_int("num_leaves", 7, 63),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 50),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq":     5,
            "lambda_l1":        trial.suggest_float("lambda_l1", 1e-3, 10, log=True),
            "lambda_l2":        trial.suggest_float("lambda_l2", 1e-3, 10, log=True),
            "is_unbalance":     True,
            "verbose":          -1,
            "random_state":     42,
        }
        aucs = []
        for tr_idx, vl_idx in GroupKFold(n_splits=3).split(df, df[target], groups):
            if training_filter is not None:
                tr_idx = np.array([i for i in tr_idx if training_filter[i]])

            X_tr = df.iloc[tr_idx][feature_cols].replace([np.inf, -np.inf], np.nan)
            y_tr = df.iloc[tr_idx][target].values
            X_vl = df.iloc[vl_idx][feature_cols].replace([np.inf, -np.inf], np.nan)
            y_vl = df.iloc[vl_idx][target].values

            if y_vl.sum() == 0 or y_tr.sum() == 0:
                continue

            train_data = lgb.Dataset(X_tr, label=y_tr)
            val_data   = lgb.Dataset(X_vl, label=y_vl, reference=train_data)
            model = lgb.train(
                params, train_data, num_boost_round=1500,
                valid_sets=[val_data], valid_names=["val"],
                callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
            )
            preds = model.predict(X_vl, num_iteration=model.best_iteration)
            aucs.append(roc_auc_score(y_vl, preds))

        return float(np.mean(aucs)) if aucs else 0.0

    return _objective

In [4]:
def train_final(best_params, target, training_filter=None, name=""):
    final_params = {
        **best_params,
        "objective":     "binary",
        "metric":        "auc",
        "learning_rate": 0.01,
        "bagging_freq":  5,
        "is_unbalance":  True,
        "verbose":       -1,
        "random_state":  42,
    }

    print(f"\nFinal training — {name} model (lr=0.01)")
    oof_preds = np.full(N, np.nan)
    fold_models, fold_aucs, fold_districts = [], [], []

    for fold, (tr_idx, vl_idx) in enumerate(GroupKFold(n_splits=3).split(df, df[target], groups), start=1):
        if training_filter is not None:
            tr_idx = np.array([i for i in tr_idx if training_filter[i]])

        X_tr = df.iloc[tr_idx][feature_cols].replace([np.inf, -np.inf], np.nan)
        y_tr = df.iloc[tr_idx][target].values
        X_vl = df.iloc[vl_idx][feature_cols].replace([np.inf, -np.inf], np.nan)
        y_vl = df.iloc[vl_idx][target].values

        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data   = lgb.Dataset(X_vl, label=y_vl, reference=train_data)
        model = lgb.train(
            final_params, train_data, num_boost_round=5000,
            valid_sets=[val_data], valid_names=["val"],
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)],
        )
        preds = model.predict(X_vl, num_iteration=model.best_iteration)
        oof_preds[vl_idx] = preds

        fold_auc = roc_auc_score(y_vl, preds) if y_vl.sum() > 0 else float("nan")
        fold_aucs.append(fold_auc)
        fold_districts.append(df.iloc[vl_idx]["district"].iloc[0])
        fold_models.append(model)
        model.save_model(f"{MODELS_DIR}/lgb_{name}_fold{fold}.txt")
        print(f"  Fold {fold} ({fold_districts[-1]:25s}): AUC = {fold_auc:.4f}  (best iter {model.best_iteration})")

    overall_auc = roc_auc_score(df[target], oof_preds)
    print(f"  Overall CV AUC: {overall_auc:.4f}")
    np.save(f"{MODELS_DIR}/oof_preds_{name}.npy", oof_preds)

    if training_filter is not None:
        full_idx = np.where(training_filter)[0]
    else:
        full_idx = np.arange(N)
    X_full = df.iloc[full_idx][feature_cols].replace([np.inf, -np.inf], np.nan)
    y_full = df.iloc[full_idx][target].values

    avg_iter = int(np.mean([m.best_iteration for m in fold_models]))
    full_data = lgb.Dataset(X_full, label=y_full)
    final_model = lgb.train(
        final_params, full_data, num_boost_round=avg_iter,
        callbacks=[lgb.log_evaluation(0)],
    )
    final_model.save_model(f"{MODELS_DIR}/lgb_{name}_final.txt")
    print(f"  Saved production model trained on full data ({avg_iter} rounds)")

    return overall_auc, oof_preds, fold_aucs

In [5]:
print("=" * 70)
print(f"Tuning continuation model ({N_TRIALS} trials)...")
print("=" * 70)

study_cont = optuna.create_study(direction="maximize",
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_cont.optimize(make_objective("flood_event"), n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest continuation CV AUC during tuning: {study_cont.best_value:.4f}")
print(f"Best params: {study_cont.best_params}")

Tuning continuation model (40 trials)...


  0%|          | 0/40 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[70]	val's auc: 0.972958
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[150]	val's auc: 0.980528
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[49]	val's auc: 0.969891
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[190]	val's auc: 0.974099
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[299]	val's auc: 0.981317
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[134]	val's auc: 0.972703
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[85]	val's auc: 0.977061
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[120]	val's auc: 0.982559
Training until validation s

In [6]:
cont_auc, cont_oof, cont_fold_aucs = train_final(
    study_cont.best_params, "flood_event", name="continuation"
)


Final training — continuation model (lr=0.01)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[482]	val's auc: 0.976741
  Fold 1 (Sindh_District           ): AUC = 0.9767  (best iter 482)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[386]	val's auc: 0.982774
  Fold 2 (Balochistan_District     ): AUC = 0.9828  (best iter 386)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[639]	val's auc: 0.972231
  Fold 3 (KP_District              ): AUC = 0.9722  (best iter 639)
  Overall CV AUC: 0.9689
  Saved production model trained on full data (502 rounds)


In [7]:
print("=" * 70)
print(f"Tuning onset model ({N_TRIALS} trials)...")
print("=" * 70)

training_filter_onset = (df["flood_state"] != "continuation").values
print(f"Training filter: {training_filter_onset.sum()} rows (continuations excluded)")

study_onset = optuna.create_study(direction="maximize",
                                   sampler=optuna.samplers.TPESampler(seed=42))
study_onset.optimize(make_objective("flood_onset", training_filter=training_filter_onset),
                     n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest onset CV AUC during tuning: {study_onset.best_value:.4f}")
print(f"Best params: {study_onset.best_params}")

Tuning onset model (40 trials)...
Training filter: 1057 rows (continuations excluded)


  0%|          | 0/40 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[75]	val's auc: 0.961386
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[134]	val's auc: 0.98921
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[124]	val's auc: 0.982871
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[61]	val's auc: 0.963537
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[197]	val's auc: 0.989051
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[251]	val's auc: 0.981652
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[8]	val's auc: 0.953553
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[83]	val's auc: 0.990858
Training until validation scor

In [8]:
onset_auc, onset_oof, onset_fold_aucs = train_final(
    study_onset.best_params, "flood_onset",
    training_filter=training_filter_onset, name="onset"
)


Final training — onset model (lr=0.01)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[2]	val's auc: 0.968529
  Fold 1 (Sindh_District           ): AUC = 0.9685  (best iter 2)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[601]	val's auc: 0.990911
  Fold 2 (Balochistan_District     ): AUC = 0.9909  (best iter 601)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[54]	val's auc: 0.979379
  Fold 3 (KP_District              ): AUC = 0.9794  (best iter 54)
  Overall CV AUC: 0.9468
  Saved production model trained on full data (219 rounds)


In [9]:
PRE_TUNE_CONT  = 0.9560
PRE_TUNE_ONSET = 0.9573

print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"""
Continuation model (target = flood_event):
   Pre-tuning AUC:    {PRE_TUNE_CONT:.4f}
   Tuned + retrained: {cont_auc:.4f}   (Δ {cont_auc - PRE_TUNE_CONT:+.4f})
   Per-fold AUCs:     {[f'{a:.4f}' for a in cont_fold_aucs]}
   Best params:       {study_cont.best_params}

Onset model (target = flood_onset):
   Pre-tuning AUC:    {PRE_TUNE_ONSET:.4f}
   Tuned + retrained: {onset_auc:.4f}   (Δ {onset_auc - PRE_TUNE_ONSET:+.4f})
   Per-fold AUCs:     {[f'{a:.4f}' for a in onset_fold_aucs]}
   Best params:       {study_onset.best_params}
""")

SUMMARY

Continuation model (target = flood_event):
   Pre-tuning AUC:    0.9560
   Tuned + retrained: 0.9689   (Δ +0.0129)
   Per-fold AUCs:     ['0.9767', '0.9828', '0.9722']
   Best params:       {'num_leaves': 20, 'min_data_in_leaf': 5, 'feature_fraction': 0.8866876305603675, 'bagging_fraction': 0.752613643399555, 'lambda_l1': 0.0023561711719811496, 'lambda_l2': 0.006584255446714257}

Onset model (target = flood_onset):
   Pre-tuning AUC:    0.9573
   Tuned + retrained: 0.9468   (Δ -0.0105)
   Per-fold AUCs:     ['0.9685', '0.9909', '0.9794']
   Best params:       {'num_leaves': 21, 'min_data_in_leaf': 22, 'feature_fraction': 0.6037596637978375, 'bagging_fraction': 0.8620980599158715, 'lambda_l1': 0.008457088322556625, 'lambda_l2': 0.2444436163607056}



In [10]:
params_summary = {
    "continuation": {"best_params": study_cont.best_params,  "cv_auc": cont_auc},
    "onset":        {"best_params": study_onset.best_params, "cv_auc": onset_auc},
}
with open(f"{MODELS_DIR}/best_params.json", "w") as f:
    json.dump(params_summary, f, indent=2)

print(f"Artifacts saved to {MODELS_DIR}/")
print(f"  lgb_continuation_fold{{1,2,3}}.txt   — CV-evaluated models")
print(f"  lgb_continuation_final.txt          — production model (full data)")
print(f"  lgb_onset_fold{{1,2,3}}.txt          — CV-evaluated models")
print(f"  lgb_onset_final.txt                  — production model (full data)")
print(f"  oof_preds_continuation.npy")
print(f"  oof_preds_onset.npy")
print(f"  best_params.json")

Artifacts saved to final_models/
  lgb_continuation_fold{1,2,3}.txt   — CV-evaluated models
  lgb_continuation_final.txt          — production model (full data)
  lgb_onset_fold{1,2,3}.txt          — CV-evaluated models
  lgb_onset_final.txt                  — production model (full data)
  oof_preds_continuation.npy
  oof_preds_onset.npy
  best_params.json
